In [1]:
import xarray as xr
import geopandas as gpd

In [5]:
import xarray as xr
import pandas as pd
import numpy as np

path = "/home/gespejogutierrez/Lisflood_climada/Zell_netcdfiles/Zell_2m_Combiprecip.nc"
ds = xr.open_dataset(path)

THRESH = 0.01

# ------------------------------------------------------------------
# 1) Create wet mask from water_depth
# ------------------------------------------------------------------
mask = ds["water_depth"] > THRESH

# ------------------------------------------------------------------
# 2) Select variables AND apply the SAME mask
# ------------------------------------------------------------------
ds_sel = ds[["water_depth", "vel_mag", "flux_mag"]].where(mask)

# ------------------------------------------------------------------
# 3) Convert to DataFrame and drop dry cells
# ------------------------------------------------------------------
Zell_wet_cells = (
    ds_sel
    .to_dataframe()
    .dropna()
    .reset_index()
)

# ------------------------------------------------------------------
# 4) Add timestep index 0..10
# ------------------------------------------------------------------
times = list(Zell_wet_cells["REFERENCE_TS"].values)
time_to_step = {t: i for i, t in enumerate(times)}
Zell_wet_cells["timestep"] = Zell_wet_cells["REFERENCE_TS"].map(time_to_step).astype("int16")

# ------------------------------------------------------------------
# 5) Check result
# ------------------------------------------------------------------
Zell_wet_cells.head(), Zell_wet_cells.shape


(         REFERENCE_TS          y          x  water_depth   vel_mag  flux_mag  \
 0 2022-05-05 17:00:00  1257999.0  2702001.0        0.020  0.005000       0.0   
 1 2022-05-05 17:00:00  1257999.0  2702025.0        0.018  0.003162       0.0   
 2 2022-05-05 17:00:00  1257999.0  2702033.0        0.035  0.012166       0.0   
 3 2022-05-05 17:00:00  1257999.0  2702061.0        0.018  0.014036       0.0   
 4 2022-05-05 17:00:00  1257999.0  2702175.0        0.026  0.002236       0.0   
 
    spatial_ref  timestep  
 0            0    -29538  
 1            0    -29538  
 2            0    -29538  
 3            0    -29538  
 4            0    -29538  ,
 (2937119, 8))

In [6]:
from sqlalchemy import create_engine

database_name = "entwicklung"
port = 5432
server = "mobidb02"
username = "gespejo"
password = "Fellbach1993#"

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{server}:{port}/{database_name}")

Zell_wet_cells.to_sql(
    "Zell_wet_cells",
    engine,
    schema="swf_modelling",
    if_exists="replace",   # use "append" later if you rerun
    index=False
)


119